# Capítulo 5 · Selección de características y comparación de modelos

**Aprendizaje supervisado · ¿Necesitamos todas las variables?**

Seleccionar características significa **conservar algunas columnas originales** y retirar otras.
Buscamos un modelo más sencillo que mantenga un desempeño útil; quitar columnas no garantiza mejorar las predicciones.
No es lo mismo que PCA, que construye nuevas variables combinando las originales.

Trabajaremos con **Wine**, incluido en scikit-learn: 178 muestras, 13 mediciones químicas
y tres clases que representan cultivares de vino. Predeciremos la clase, no la calidad ni el precio.
Hay suficientes columnas para comparar técnicas y pocas filas para que las búsquedas sean manejables.
No hace falta descargar archivos ni conectarse a Internet.

### Objetivos

- Distinguir filtros, métodos de búsqueda e integrados.
- Aplicar cada técnica y ver qué columnas conserva.
- Comparar las columnas elegidas por cada técnica.
- Entrenar distintos modelos con cada subconjunto y con todas las columnas.
- Elegir con validación cruzada en train y evaluar una sola vez en test.

### Índice

Preparación común: diagnóstico, limpieza y partición de datos.

1. Filtro de varianza
2. Filtro de correlación entre características
3. Selección univariada: ANOVA
4. Selección hacia adelante (*forward selection*)
5. Selección hacia atrás (*backward elimination*)
6. Eliminación recursiva: RFE
7. Selección con Lasso
8. Extensión: búsqueda exhaustiva

9. Comparar las características seleccionadas
10. Entrenar SVC, regresión logística y Random Forest

Cierre: evaluación final y preguntas para discutir.
La búsqueda exhaustiva puede revisarse en una segunda sesión.

Fuente del dataset: [Wine en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html).

In [ ]:
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    RFE, SelectKBest, SequentialFeatureSelector,
    VarianceThreshold, f_classif,
)
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [ ]:
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
N_FEATURES = 6

---
## Preparación común

### Diagnóstico y limpieza simple

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
np.isinf(df).sum()

In [ ]:
df.duplicated().sum()

Aplicamos la limpieza en pasos separados. En Wine no se pierden filas porque ya está limpio.

In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df.dropna()
df.shape

### Entrenamiento y prueba: 80-20

Hacemos una sola división:

- **80 % para entrenamiento:** explorar, seleccionar características, entrenar y comparar candidatos.
- **20 % para prueba (test):** evaluar el modelo elegido al final.

`test_size=0.2` reserva el 20 % para test. `stratify=y` mantiene aproximadamente la proporción
de las clases y `random_state=42` permite repetir la partición.
La columna `target` es la respuesta y no se incluye entre las características.

La búsqueda de hiperparámetros usará tres particiones de validación cruzada dentro del 80 % de train.
No creamos un conjunto de validación separado. Test queda fuera de todas las decisiones.

In [ ]:
X = df.drop(columns="target")
y = df["target"].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

In [ ]:
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
y_train.value_counts().sort_index()

In [ ]:
X_train.describe().T[["min", "mean", "std", "max"]].round(2)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X_train.corr(), cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Wine · Correlación entre mediciones de entrenamiento")
plt.tight_layout()
plt.show()

In [ ]:
selected_columns_all = X_train.columns.tolist()
selected_columns_all

---
## 1. Filtro de varianza

**Idea:** una columna que siempre tiene el mismo valor no ayuda a distinguir unas muestras de otras.
`VarianceThreshold(threshold=0)` elimina exclusivamente columnas constantes, sin consultar `y`.

Usamos el umbral cero porque las mediciones tienen unidades distintas: comparar directamente sus
varianzas con un umbral positivo puede confundir escala con información. El filtro va antes del escalado.
Wine no tiene columnas constantes; esperamos conservar las 13. No necesitamos forzar una eliminación.

**Límite:** poca variación no significa necesariamente poca utilidad predictiva.

Referencia: [VarianceThreshold](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.VarianceThreshold.html).

In [ ]:
variance_selector = VarianceThreshold(threshold=0)

In [ ]:
variance_selector.fit(X_train)

In [ ]:
selected_columns_variance = X_train.columns[variance_selector.get_support()].tolist()
selected_columns_variance

---
## 2. Filtro de correlación entre características

**Idea:** si dos columnas se parecen mucho, probamos conservar solo una.
Usamos la correlación de Pearson **entre características**, no entre las características y la clase numérica.
El signo no cambia la redundancia: una correlación de -0.95 también es fuerte, por eso tomamos el valor absoluto.

Regla de esta práctica: recorrer las columnas en su orden original y conservar una si su correlación
absoluta con todas las ya conservadas es menor que 0.85. El orden y el umbral influyen en el resultado.
No demuestra que la columna descartada sea inútil ni detecta todas las relaciones no lineales.

In [ ]:
correlation = X_train.corr().abs()
correlation.round(2)

Recorremos las columnas. Conservamos una columna si todas sus correlaciones con las ya elegidas
son menores que 0.85. `.all()` comprueba que se cumpla para todas; al principio la lista está vacía
y se conserva la primera columna.

In [ ]:
selected_columns_correlation = []

for column in X_train.columns:
    correlations = correlation.loc[column, selected_columns_correlation]
    if (correlations < 0.85).all():
        selected_columns_correlation.append(column)

In [ ]:
selected_columns_correlation

---
## 3. Selección univariada: ANOVA

**Idea:** evaluar cada característica por separado y conservar las seis con mayor puntuación.
`f_classif` compara cuánto cambian las medias de una medición entre clases frente a su variación dentro de las clases.
Una puntuación F mayor sugiere más separación entre grupos para esa medición.

`SelectKBest` ordena esas puntuaciones y conserva las primeras `k`.
Aquí lo usamos como filtro exploratorio; no interpretaremos sus p-valores como evidencia científica.

**Límite:** dos variables pueden ser útiles juntas aunque ninguna destaque por separado.
También puede conservar varias columnas redundantes porque no compara unas características con otras.

Referencia: [SelectKBest](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectKBest.html).

In [ ]:
anova_selector = SelectKBest(score_func=f_classif, k=N_FEATURES)

In [ ]:
anova_selector.fit(X_train, y_train)

In [ ]:
selected_columns_anova = X_train.columns[anova_selector.get_support()].tolist()
selected_columns_anova

In [ ]:
anova_scores = pd.Series(anova_selector.scores_, index=X_train.columns)
anova_scores.sort_values(ascending=False)

In [ ]:
anova_scores.sort_values().plot.barh(figsize=(8, 5))
plt.title("ANOVA · Puntuación individual de cada medición en train")
plt.xlabel("Puntuación F")
plt.tight_layout()
plt.show()

---
## 4. Selección hacia adelante · Forward selection

El selector empieza con la columna que obtiene la mayor accuracy media en validación cruzada. Luego prueba añadir cada una de las restantes. Acepta la mejor incorporación si la accuracy aumenta al menos `1e-6`. Cuando la mejora queda por debajo de ese valor, se detiene.

La cantidad de columnas se decide durante el ajuste con `n_features_to_select="auto"` y `tol`. [Documentación de SequentialFeatureSelector](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html).

Preparamos el SVC que evaluará los subconjuntos y tres particiones de entrenamiento.
Esta validación cruzada es **interna al selector**; solo usa `X_train` e `y_train`; no usa test.
El pipeline ajusta el escalado dentro de cada partición interna.

In [ ]:
selection_model = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1)),
])

In [ ]:
selection_cv = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE,
)

In [ ]:
forward_selector = SequentialFeatureSelector(
    selection_model,
    n_features_to_select="auto",
    tol=1e-6,
    direction="forward",
    scoring="accuracy",
    cv=selection_cv,
)

In [ ]:
forward_selector.fit(X_train, y_train)

In [ ]:
selected_columns_forward = X_train.columns[forward_selector.get_support()].tolist()
selected_columns_forward

---
## 5. Selección hacia atrás · Backward elimination

El selector parte de las 13 columnas y elige la primera que puede retirar con la mayor accuracy media en validación cruzada. Ese resultado es su referencia para el siguiente paso.

Con `tol=0.0`, sigue retirando una columna por paso mientras la accuracy se mantenga o mejore. Si todas las eliminaciones posibles reducen la puntuación respecto al paso anterior, se detiene.

In [ ]:
backward_selector = SequentialFeatureSelector(
    selection_model,
    n_features_to_select="auto",
    tol=0.0,
    direction="backward",
    scoring="accuracy",
    cv=selection_cv,
)

In [ ]:
backward_selector.fit(X_train, y_train)

In [ ]:
selected_columns_backward = X_train.columns[backward_selector.get_support()].tolist()
selected_columns_backward

Comparamos las listas. Convertirlas a `set` permite ver sus coincidencias y diferencias.

In [ ]:
print("En ambos:", sorted(set(selected_columns_forward) & set(selected_columns_backward)))
print("Solo forward:", sorted(set(selected_columns_forward) - set(selected_columns_backward)))
print("Solo backward:", sorted(set(selected_columns_backward) - set(selected_columns_forward)))

---
## 6. Eliminación recursiva · RFE

**Idea:** entrenar un modelo, retirar la variable con menor peso y volver a entrenar hasta conservar seis.

Backward prueba cada posible eliminación con validación; **RFE utiliza los pesos del modelo ajustado**.
Por eso suele necesitar menos entrenamientos.

El SVC con RBF no tiene un coeficiente por característica. Usaremos un **SVC lineal para seleccionar**,
que sí tiene `coef_`. Guardaremos sus columnas para la comparación final de modelos.
Primero estandarizamos las mediciones usando únicamente train para que sus pesos sean comparables.
En multiclase, RFE combina los coeficientes al cuadrado al ordenar las variables.

**Límite:** una variable poco útil para una frontera lineal podría ser útil para una no lineal.

Referencia: [RFE](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html).

In [ ]:
selection_scaler = StandardScaler()
X_train_scaled = selection_scaler.fit_transform(X_train)

In [ ]:
rfe_selector = RFE(
    estimator=SVC(kernel="linear", C=1),
    n_features_to_select=N_FEATURES,
    step=1,
)

In [ ]:
rfe_selector.fit(X_train_scaled, y_train)

In [ ]:
selected_columns_rfe = X_train.columns[rfe_selector.get_support()].tolist()
selected_columns_rfe

El rango `1` identifica las variables conservadas. Los demás rangos corresponden a variables descartadas.

In [ ]:
rfe_ranking = pd.Series(rfe_selector.ranking_, index=X_train.columns)
rfe_ranking.sort_values()

## 7. Selección con Lasso

Lasso es una regresión que puede llevar algunos coeficientes a cero. Usaremos las mediciones estandarizadas de entrenamiento. Fijamos `alpha=0.05` para este ejemplo. Un valor mayor penaliza más los coeficientes.

Wine tiene tres clases. Para cada una creamos una columna que vale 1 cuando la muestra pertenece a esa clase y 0 en los demás casos. Así Lasso ajusta una regresión por clase y podemos ver qué mediciones tienen peso en cada una.

In [ ]:
class_indicators = pd.get_dummies(y_train, prefix="clase", dtype=int)
class_indicators.head()

In [ ]:
lasso = Lasso(alpha=0.05)
lasso.fit(X_train_scaled, class_indicators)

In [ ]:
lasso_weights = pd.DataFrame(lasso.coef_, index=class_indicators.columns, columns=X_train.columns)
lasso_weights.T

Conservamos una característica si su coeficiente es distinto de cero en al menos una clase. En la siguiente celda, `.any()` comprueba esa condición para cada característica.

In [ ]:
keep = (lasso_weights != 0).any()
selected_columns_lasso = X_train.columns[keep].tolist()
selected_columns_lasso

---
## 8. Extensión: búsqueda exhaustiva

**Idea:** probar todos los subconjuntos no vacíos de las columnas candidatas.
Con `p` columnas existen `2**p - 1` subconjuntos. Con las 13 de Wine serían 8191,
y cada uno necesita varios entrenamientos para evaluarse.

Para mostrarlo de forma rápida declaramos cinco candidatas: alcohol, ácido málico, flavonoides,
intensidad de color y prolina. Probamos sus 31 subconjuntos con el SVC fijo y validación cruzada sobre train.
Las candidatas se fijaron de antemano, sin consultar test.

La búsqueda es exhaustiva **solo entre esas cinco columnas y para ese SVC fijo**.
Su espacio de búsqueda es distinto del de los métodos que consideran las 13 variables.

In [ ]:
def subset_accuracy(columns):
    scores = cross_val_score(
        selection_model, X_train[columns], y_train,
        cv=selection_cv, scoring="accuracy",
    )
    return scores.mean()

In [ ]:
exhaustive_candidates = [
    "alcohol", "malic_acid", "flavanoids", "color_intensity", "proline",
]
2 ** len(exhaustive_candidates) - 1

`combinations` construye los grupos de una, dos, tres, cuatro y cinco columnas.
Guardamos su puntuación para poder ver la tabla de resultados.

In [ ]:
exhaustive_results = []

for size in range(1, len(exhaustive_candidates) + 1):
    for subset in combinations(exhaustive_candidates, size):
        columns = list(subset)
        score = subset_accuracy(columns)
        exhaustive_results.append({
            "columnas": columns,
            "variables": len(columns),
            "accuracy_cv": score,
        })

In [ ]:
exhaustive_table = pd.DataFrame(exhaustive_results)
exhaustive_table = exhaustive_table.sort_values(
    ["accuracy_cv", "variables"], ascending=[False, True],
)
exhaustive_table.head()

In [ ]:
selected_columns_exhaustive = exhaustive_table.iloc[0]["columnas"]
selected_columns_exhaustive

## 9. ¿Qué características eligió cada técnica?

Cada fila de la gráfica representa una medición de Wine. Un cuadro azul indica que el selector de esa columna la conservó. La primera columna muestra la referencia con todas las mediciones.

In [ ]:
feature_sets = {
    "Sin selección": selected_columns_all,
    "Varianza cero": selected_columns_variance,
    "Correlación": selected_columns_correlation,
    "ANOVA": selected_columns_anova,
    "Hacia adelante": selected_columns_forward,
    "Hacia atrás": selected_columns_backward,
    "RFE": selected_columns_rfe,
    "Lasso": selected_columns_lasso,
    "Exhaustiva (5 candidatas)": selected_columns_exhaustive,
}

In [ ]:
feature_matrix = pd.DataFrame(index=X_train.columns)
for method, columns in feature_sets.items():
    feature_matrix[method] = X_train.columns.isin(columns)

In [ ]:
plt.figure(figsize=(12, 7))
sns.heatmap(feature_matrix.astype(int), annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Características seleccionadas · 1: conservada, 0: descartada")
plt.xlabel("Técnica de selección")
plt.ylabel("Característica")
plt.tight_layout()
plt.show()

## 10. ¿Qué accuracy obtuvo cada modelo?

Usaremos las mismas tres particiones de entrenamiento para todos los modelos. Probaremos tres valores de `C` para SVC y regresión logística, y tres profundidades para Random Forest. La tabla final mostrará la mejor accuracy media de cada combinación de selector y modelo.

Como las columnas ya se eligieron con todo el entrenamiento, estas puntuaciones pueden ser optimistas. El 20 % de prueba queda reservado para evaluar la combinación elegida.

In [ ]:
model_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
c_values = [0.1, 1, 10]

SVC y regresión logística aprenden el escalado dentro de cada partición. Random Forest trabaja directamente con las mediciones.

In [ ]:
svc = Pipeline([("escala", StandardScaler()), ("svc", SVC(kernel="rbf"))])
logistic = Pipeline([("escala", StandardScaler()), ("logistic", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
forest = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

In [ ]:
candidates = [
    ("SVC", svc, {"svc__C": c_values}),
    ("Regresión logística", logistic, {"logistic__C": c_values}),
    ("Random Forest", forest, {"max_depth": [3, 5, None]}),
]

Para cada lista de columnas, `GridSearchCV` elige el mejor ajuste de cada modelo usando la accuracy media de validación cruzada.

In [ ]:
results = []
trained_models = {}
for selector, columns in feature_sets.items():
    for name, model, grid in candidates:
        search = GridSearchCV(model, grid, cv=model_cv, scoring="accuracy")
        search.fit(X_train[columns], y_train)
        trained_models[(selector, name)] = search.best_estimator_
        results.append({
            "selector": selector, "variables": len(columns), "modelo": name,
            "accuracy_cv": search.best_score_, "parámetros": search.best_params_,
        })

### Comparación de resultados

Cada fila corresponde a un conjunto de columnas. Lee la tabla de arriba abajo para comparar un modelo con su resultado en «Sin selección», o de izquierda a derecha para comparar los tres modelos.

In [ ]:
results_table = pd.DataFrame(results)
model_comparison = results_table.pivot(index=["selector", "variables"], columns="modelo", values="accuracy_cv")
model_comparison = model_comparison.reindex(feature_sets.keys(), level="selector")
model_comparison[["SVC", "Regresión logística", "Random Forest"]].round(4)

## Evaluación final

Elegimos la mayor accuracy media con los valores completos, antes de redondear. Si hay empate, preferimos el subconjunto con menos variables. Usamos el 20 % de prueba para medir cuántos vinos clasifica correctamente esa combinación.

In [ ]:
ranked = results_table.sort_values(["accuracy_cv", "variables"], ascending=[False, True], kind="stable")
best_result = ranked.iloc[0]

In [ ]:
winner_selector = best_result["selector"]
winner_name = best_result["modelo"]
winner_columns = feature_sets[winner_selector]
print(winner_selector, "·", winner_name, "·", best_result["parámetros"])

In [ ]:
final_model = trained_models[(winner_selector, winner_name)]

In [ ]:
y_pred = final_model.predict(X_test[winner_columns])

In [ ]:
test_accuracy = accuracy_score(y_test, y_pred)
test_accuracy

In [ ]:
print(classification_report(
    y_test, y_pred, labels=[0, 1, 2], target_names=wine.target_names,
    digits=3, zero_division=0,
))

In [ ]:
matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

In [ ]:
sns.heatmap(
    matrix, annot=True, fmt="d", cmap="Blues",
    xticklabels=wine.target_names, yticklabels=wine.target_names,
)
plt.title(f"Wine · {winner_selector} + {winner_name}")
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
plt.tight_layout()
plt.show()

### Chuleta de técnicas

| Técnica | ¿Qué criterio usa? | ¿Necesita la clase? | Precaución principal |
|---|---|---|---|
| Varianza cero | Variación de cada columna | No | No elimina todas las variables poco útiles |
| Correlación | Redundancia lineal entre columnas | No | Depende del orden y del umbral |
| ANOVA | Separación individual entre clases | Sí | No evalúa combinaciones |
| Forward | Accuracy al incorporar una variable | Sí | No reconsidera incorporaciones anteriores |
| Backward | Accuracy al retirar una variable | Sí | Necesita muchos entrenamientos |
| RFE | Pesos del modelo y eliminación sucesiva | Sí | Depende del modelo que produce los pesos |
| Lasso | Coeficientes reducidos a cero | Sí | Depende del escalado y la regularización |
| Exhaustiva | Todos los subconjuntos de las candidatas | Sí | El costo crece exponencialmente |